# 03. EXPLORATORY DATA ANALYSIS - Final Data Cleaning & Feature Set. Final data cleaning, consolidated feature set, category grouping, log transformations, scaling, and preparing the dataset for modeling.

# 1. Importing libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import scipy.stats as stats
from matplotlib.gridspec import GridSpec
import matplotlib.patches as patches
from sklearn.preprocessing import RobustScaler

# 2. Importing dataset

In [2]:
BASE_DIR = os.path.dirname(os.getcwd())
DATA_PATH = os.path.join(BASE_DIR, "data", "House_Rate_Data.csv")
df = pd.read_csv(DATA_PATH)
df_final = df.copy()

# 3. Basic data overview

In [3]:
#a) structure:
print("=====  STRUCTURE =====")

print("\Info:")
print(df_final.info())

print("\nShape (rows, columns):")
print(df_final.shape)

print("\nHead:")
display(df_final.head())

print("\nDtypes:")
print(df_final.dtypes)

#b) quality

print("\n===== QUALITY =====")

print("\nDescribe:")
display(df_final.describe())

print("\nMissing values (count):")
print(df_final.isnull().sum)

print("\nMissing values (%):")
print(df_final.isnull().mean() * 100)

#c) numerical and categorical columns comparison

print("\nNumerical Columns:")
numerical_cols = df_final.select_dtypes(include=['int64', 'float64']).columns
print(list(numerical_cols))

print("\Categorical Columns:")
categorical_cols = df_final.select_dtypes(include=object).columns
print(list(categorical_cols))

=====  STRUCTURE =====
\Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            4600 non-null   object 
 1   Rates or Price  4600 non-null   float64
 2   Bedrooms        4600 non-null   int64  
 3   Bathrooms       4600 non-null   float64
 4   Sqft_Living     4600 non-null   int64  
 5   Sqft_lot        4600 non-null   int64  
 6   Floors          4600 non-null   float64
 7   Waterfront      4600 non-null   int64  
 8   view            4600 non-null   int64  
 9   Condition       4600 non-null   int64  
 10  Sqft_Above      4600 non-null   int64  
 11  Sqft_Basement   4600 non-null   int64  
dtypes: float64(3), int64(8), object(1)
memory usage: 431.4+ KB
None

Shape (rows, columns):
(4600, 12)

Head:


,Date,Rates or Price,Bedrooms,Bathrooms,Sqft_Living,Sqft_lot,Floors,Waterfront,view,Condition,Sqft_Above,Sqft_Basement
0,02-05-2014,313000.0,3,1.50,1340,7912,1.5,0,0,3,1340,0
1,02-05-2014,2384000.0,5,2.50,3650,9050,2.0,0,4,5,3370,280
2,02-05-2014,342000.0,3,2.00,1930,11947,1.0,0,0,4,1930,0
3,02-05-2014,420000.0,3,2.25,2000,8030,1.0,0,0,4,1000,1000
4,02-05-2014,550000.0,4,2.50,1940,10500,1.0,0,0,4,1140,800



Dtypes:
Date               object
Rates or Price    float64
Bedrooms            int64
Bathrooms         float64
Sqft_Living         int64
Sqft_lot            int64
Floors            float64
Waterfront          int64
view                int64
Condition           int64
Sqft_Above          int64
Sqft_Basement       int64
dtype: object

===== QUALITY =====

Describe:


,Rates or Price,Bedrooms,Bathrooms,Sqft_Living,Sqft_lot,Floors,Waterfront,view,Condition,Sqft_Above,Sqft_Basement
count,4.600000e+03,4600.000000,4600.000000,4600.000000,4.600000e+03,4600.000000,4600.000000,4600.000000,4600.000000,4600.000000,4600.000000
mean,5.519630e+05,3.400870,2.160815,2139.346957,1.485252e+04,1.512065,0.007174,0.240652,3.451739,1827.265435,312.081522
std,5.638347e+05,0.908848,0.783781,963.206916,3.588444e+04,0.538288,0.084404,0.778405,0.677230,862.168977,464.137228
min,0.000000e+00,0.000000,0.000000,370.000000,6.380000e+02,1.000000,0.000000,0.000000,1.000000,370.000000,0.000000
25%,3.228750e+05,3.000000,1.750000,1460.000000,5.000750e+03,1.000000,0.000000,0.000000,3.000000,1190.000000,0.000000
50%,4.609435e+05,3.000000,2.250000,1980.000000,7.683000e+03,1.500000,0.000000,0.000000,3.000000,1590.000000,0.000000
75%,6.549625e+05,4.000000,2.500000,2620.000000,1.100125e+04,2.000000,0.000000,0.000000,4.000000,2300.000000,610.000000
max,2.659000e+07,9.000000,8.000000,13540.000000,1.074218e+06,3.500000,1.000000,4.000000,5.000000,9410.000000,4820.000000



Missing values (count):
<bound method DataFrame.sum of        Date  Rates or Price  Bedrooms  Bathrooms  Sqft_Living  Sqft_lot  \
0     False           False     False      False        False     False   
1     False           False     False      False        False     False   
2     False           False     False      False        False     False   
3     False           False     False      False        False     False   
4     False           False     False      False        False     False   
...     ...             ...       ...        ...          ...       ...   
4595  False           False     False      False        False     False   
4596  False           False     False      False        False     False   
4597  False           False     False      False        False     False   
4598  False           False     False      False        False     False   
4599  False           False     False      False        False     False   

      Floors  Waterfront   view  Condition 

# 4. Data Cleaning — Removing Outliers, Invalid Records, and Dropping Redundant Columns

In [5]:
#a.deletion outliers - because there are only 3 outliners from 4600 records and they have influence on model
df_final = df_final[df_final['Rates or Price'] <= 5000000]

#b.deletion records with 0 bedroom and 0 bathroom - there isn't house with zero bedrooms or bathrooms Nowadays
df_final = df_final[(df_final['Bedrooms'] > 0) & (df_final['Bathrooms'] > 0)]

#c. deletion columns: 'Data' and "Sqft_Above"
#  - the column 'Data' has nothing to do with HouseDataset (this is only input record to the system)
#  ,on the other hand 'Sqft_Above' is redundant feature to 'Sqft_Living' so it can cause exist multilinear issue by that
df_final = df_final.drop(columns=['Date', 'Sqft_Above'])

# 5. Feature Engineering - adding new columns

In [ ]:
#a. adding 'Has_Basement'. The new column is created base on 'Sqft_Basement'
# , if in this column is any sqft that mean we have got basement in the house.
df_final['Has_Basement'] = (df_final['Sqft_Basement'] > 0).astype(int)

#b. adding 'Sqft_per_Floor'. As correlation show, this floor seems to be relevant for future modelling
df_final['Sqft_per_Floor'] = df_final['Sqft_Living'] / df_final['Floors']

#c. creating column 'View_Type', which show quality of view in House
# -For linear models, this can be an advantage.0 = no view, 1–3 = standard view, 4 = premium view.
def group_view(df):
    df_final["View_Type"] = df_final["view"].apply(
        lambda x: 0 if x == 0 else (2 if x == 4 else 1)
    )
    return df

#d. creating 'Condition_Status" column based on 'Condition'.
#  It reduces noise, smooths the relationship, improves interpretability, and decreases variance
def map_condition(x):
    if x in [1, 2]:
        return 'Poor'
    elif x in [3, 4]:
        return 'Normal'
    else:
        return 'Good'
    
df_final['Condition_Status'] = df_final['Condition'].apply(map_condition)

# 6. Feature Engineering: Grouping Rare Categories (merging categories with very low frequency):
#   Extreme values have very few observations,
#   Models do not perform well with categories that have very low frequency.
#   Merging rare categories improves model stability

In [ ]:
#a. linking values bathroom > 4.75 (e.g.: bathroom 8 has only one record)
df_final['Bathrooms_grouped'] = df_final['Bathrooms'].apply(
    lambda x: '4.75+' if x > 4.75 else str(x)
)

#a_v2 - linking and treat as numeric values
#df_eng['Bathrooms'] = df_eng['Bathrooms'].clip(upper=4.75) 

#b. merge 3.0 and 3.5 floors (as floor 3.5 has only 2 records)
df_final['Floors_grouped'] = df_final['Floors'].apply(
    lambda x: '3+' if x >= 3.0 else str(x) 
)

#b_v2 - Merging Floors 3.0 and 3.5 (Numerical). We simply treat 3.5 as 3.0. 
    #   It stays a number, avoiding unnecessary One-Hot Encoding.
#df_eng['Floors'] = df_eng['Floors'].replace({3.5: 3.0})

# 7. Overview on Dataset after changes

In [12]:
print("\nInfo:")
print(df_final.info())

print("\nHead:")
display(df_final.head())


Info:
<class 'pandas.core.frame.DataFrame'>
Index: 4595 entries, 0 to 4599
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Rates or Price     4595 non-null   float64
 1   Bedrooms           4595 non-null   int64  
 2   Bathrooms          4595 non-null   float64
 3   Sqft_Living        4595 non-null   int64  
 4   Sqft_lot           4595 non-null   int64  
 5   Floors             4595 non-null   float64
 6   Waterfront         4595 non-null   int64  
 7   view               4595 non-null   int64  
 8   Condition          4595 non-null   int64  
 9   Sqft_Basement      4595 non-null   int64  
 10  Has_Basement       4595 non-null   int64  
 11  Sqft_per_Floor     4595 non-null   float64
 12  View_Exposition    4595 non-null   object 
 13  Condition_Status   4595 non-null   object 
 14  Bathrooms_grouped  4595 non-null   object 
 15  Floors_grouped     4595 non-null   object 
dtypes: float64(4), int64(8

,Rates or Price,Bedrooms,Bathrooms,Sqft_Living,Sqft_lot,Floors,Waterfront,view,Condition,Sqft_Basement,Has_Basement,Sqft_per_Floor,View_Exposition,Condition_Status,Bathrooms_grouped,Floors_grouped
0,313000.0,3,1.50,1340,7912,1.5,0,0,3,0,0,893.333333,Normal,Normal,1.5,1.5
1,2384000.0,5,2.50,3650,9050,2.0,0,4,5,280,1,1825.000000,Premium,Good,2.5,2.0
2,342000.0,3,2.00,1930,11947,1.0,0,0,4,0,0,1930.000000,Normal,Normal,2.0,1.0
3,420000.0,3,2.25,2000,8030,1.0,0,0,4,1000,1,2000.000000,Normal,Normal,2.25,1.0
4,550000.0,4,2.50,1940,10500,1.0,0,0,4,800,1,1940.000000,Normal,Normal,2.5,1.0
